# Hala H41 private original H23 CUDA lifecycle research
Unmodified historically reviewed 53-file source at exact SHA. Private Kaggle Batch free GPU only. No native OS or commercial uses. A completed notebook is NOT CUDA cancellation qualification.


In [ ]:
import os,sys,platform,subprocess,hashlib,json,time,pathlib,importlib.metadata
from pathlib import Path
start=time.monotonic()
working=Path('/kaggle/working')
historical=working/'hala-h41-h23-original-source'
run_path=working/'hala-h41-h23-run'
receipt=working/'h41-h23-launch.json'
if (platform.system()!='Linux' or os.environ.get('KAGGLE_KERNEL_RUN_TYPE')!='Batch'
    or not working.is_dir() or os.environ.get('HALA_REMOTE_DEVELOPMENT_ALLOWED')=='1'):
    raise RuntimeError('H41 requires actual Kaggle Batch Linux, no inherited opt-in')
if historical.exists() or run_path.exists() or receipt.exists():
    raise FileExistsError('H41 refuses overwrite or resume')
if subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True,timeout=18).returncode!=0:
    raise RuntimeError('H41 requires a real free Kaggle GPU allocation')
os.environ.update(HALA_REMOTE_DEVELOPMENT_ALLOWED='1',
 HF_HUB_DISABLE_TELEMETRY='1',HF_HUB_DISABLE_IMPLICIT_TOKEN='1',
 DO_NOT_TRACK='1',WANDB_DISABLED='true',TOKENIZERS_PARALLELISM='false')
# Read-only historical Git checkout: never patch source, freeze, permissions or model.
subprocess.run(['git','clone','--quiet','--filter=blob:none','--no-checkout',
                '--depth','1','--single-branch','--branch','ps/h41-h23-real-cuda-eval',
                'https://github.com/TheHalfMoon/Hala.git',str(historical)],
                timeout=240,check=True)
subprocess.run(['git','fetch','--quiet','--depth','1','origin','f104e3e68fe378faf65c63248cb239bd2d171c28'],
               cwd=historical,timeout=220,check=True)
subprocess.run(['git','checkout','--quiet','--detach','f104e3e68fe378faf65c63248cb239bd2d171c28'],
               cwd=historical,timeout=90,check=True)
def git(*args):
    return subprocess.check_output(['git',*args],cwd=historical,text=True,timeout=30).strip()
if git('rev-parse','HEAD')!='f104e3e68fe378faf65c63248cb239bd2d171c28':
    raise RuntimeError('historical commit mismatch')
if git('status','--porcelain','--untracked-files=all'):
    raise RuntimeError('historical source is not clean')
freeze=(historical/'experiments/serving-lifecycle/source-freeze.json').read_bytes()
if hashlib.sha256(freeze).hexdigest()!='4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a':
    raise RuntimeError('original H23 source freeze changed')
# Admission checks original frozen source files, BEFORE installing dependencies.
sys.path.insert(0,str(historical))
from evaluation.serving_lifecycle import verify_sources
verify_sources('4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a')
# Only exact prescribed Pillow version is installed in the isolated ephemeral Kaggle VM.
if importlib.metadata.version('Pillow')!='11.3.0':
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                    '--no-input','--no-deps','--quiet','Pillow==11.3.0'],
                   timeout=180,check=True)
import PIL
from PIL import features
font=Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
pin_sha='ae7b7855e115a5966d8b1b3f80f254ccc117ec86f9965e202ee2940453837280'
if not font.is_file() or hashlib.sha256(font.read_bytes()).hexdigest()!=pin_sha:
    raise RuntimeError('original H23 font not matched; fail closed')
import torch,transformers
if (importlib.metadata.version('Pillow')!='11.3.0'
    or not features.check_feature('raqm')
    or torch.__version__.split('+')[0]!='2.11.0'
    or transformers.__version__!='5.16.1'
    or not torch.cuda.is_available()):
    raise RuntimeError('original exact H23 runtime pins or RAQM/CUDA not met')
from evaluation.serving_lifecycle import preflight
preflight('kaggle',str(font),'4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a','f104e3e68fe378faf65c63248cb239bd2d171c28')
info={'schema':'hala.h41.h23_original_runtime.v1',
 'original_source_commit':git('rev-parse','HEAD'),
 'original_freeze_sha256':'4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a',
 'historical_source_unmodified':True,
 'model_id':'Qwen/Qwen3-VL-2B-Instruct',
 'torch':torch.__version__,'transformers':transformers.__version__,
 'Pillow':importlib.metadata.version('Pillow'),
 'raqm':features.check_feature('raqm'),
 'font_sha256':hashlib.sha256(font.read_bytes()).hexdigest(),
 'cuda_devices':[str(torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())],
 'gpu_observed':subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True,timeout=18).stdout,
 'model_qualified':False,'actual_cuda_cancel_qualified':False,
 'native_execution':False,'noncommercial_research_only':True}
receipt.write_text(json.dumps(info,sort_keys=True,indent=2)+'\n',encoding='utf-8')
print('H41_ORIGINAL_H23_HARD_GATES_PASS','f104e3e68fe378faf65c63248cb239bd2d171c28','4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a',flush=True)


In [ ]:
import os,signal,subprocess,sys,json,time,hashlib
from pathlib import Path
target=Path('/kaggle/working/hala-h41-h23-run')
receipt=Path('/kaggle/working/h41-h23-launch.json')
info=json.loads(receipt.read_text('utf-8'))
log=Path('/kaggle/working/h41-h23-operator.log')
cmd=[sys.executable,'-u','-m','evaluation.serving_lifecycle',
  '--provider','kaggle','--freeze-sha256','4c52241c1312648ae08d2530f709b45ed3fec41ae8924d2a03df5b431fcefc8a',
  '--source-commit','f104e3e68fe378faf65c63248cb239bd2d171c28',
  '--font','/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
  '--output',str(target),'--faults']
with log.open('xb') as output:
    proc=subprocess.Popen(cmd,cwd=historical,stdout=output,
                          stderr=subprocess.STDOUT,start_new_session=True)
    timed_out=False
    try:
        code=proc.wait(timeout=1000)
    except subprocess.TimeoutExpired:
        timed_out=True
        os.killpg(proc.pid,signal.SIGTERM)
        try:code=proc.wait(timeout=12)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid,signal.SIGKILL)
            code=proc.wait(timeout=12)
info.update(operator_exit_code=code,operator_timed_out=timed_out,
            elapsed_seconds=time.monotonic()-start,
            operator_stdout_sha256=hashlib.sha256(log.read_bytes()).hexdigest(),
            operator_output_available=target.exists(),
            state='incomplete' if timed_out or code!=0 else 'completed_unqualified',
            model_qualified=False,actual_cuda_cancel_qualified=False,
            native_execution=False)
receipt.write_text(json.dumps(info,sort_keys=True,indent=2)+'\n',encoding='utf-8')
print('H41_ORIGINAL_H23_OPERATOR_EXIT',code,'TIMEOUT',timed_out,flush=True)
print('H41_FINAL_RECEIPT_SHA256',hashlib.sha256(receipt.read_bytes()).hexdigest(),flush=True)
print('H41_NO_QUALIFICATION_BEFORE_INDEPENDENT_EVIDENCE_REPLAY',flush=True)
if timed_out or code!=0:raise RuntimeError('Original H23 remote lifecycle incomplete; retain raw evidence')
